# Zero-shot IGT with Bolmo 1B (Korean / Ainu)

Two settings from the agenda, both **one model call**:
- **all-in-one**: predict transliteration and gloss together
- **step-by-step**: the same prompt asks for Step 1 (transliteration), then Step 2 (gloss based on Step 1)

Input = original script + English translation. Output = transliteration + gloss.
Prompts: `zero_shot/prompts/` (Korean) and `zero_shot/prompts/ainu/`. Logic: `zero_shot/run_zero_shot.py`.

Before running: **Runtime -> Change runtime type -> T4 GPU**.

## 1. Install (once per runtime), then Runtime -> Restart session

In [ ]:
!pip install "transformers==4.57.3" "xlstm==2.0.4" datasets pandas accelerate

## 2. Get the code and check the environment

In [ ]:
import os, sys, importlib, importlib.util
REPO = "/content/Korean-IGT-Bolmo"
if not os.path.isdir(REPO):
    !git clone https://github.com/JingyiShao-Gina/Korean-IGT-Bolmo.git {REPO}
else:
    !git -C {REPO} pull
os.chdir(REPO)
sys.path.insert(0, REPO + "/zero_shot")
!git log --oneline -1

import transformers
print("transformers:", transformers.__version__, "(need 4.57.3)")
print("xlstm installed:", importlib.util.find_spec("xlstm") is not None)

## 3. Choose the language and preview the prompts (no model needed)

In [ ]:
LANG = "ainu"   # "ainu" or "korean"

import json, pandas as pd
import run_zero_shot as rz
importlib.reload(rz)
cfg = json.load(open("configs/zero_shot.json"))
L = cfg["languages"][LANG]
T = {k: rz.load_prompt(v) for k, v in L["prompts"].items()}
ex = rz.load_rows("dev", 1, rz.ROOT / L["dataset_manifest"])[0]
for key in ("all_in_one", "step_by_step"):
    print(f"----- {key} -----")
    print(rz.render(T[key], original_script=ex["text"], translation=ex["translation"]))
print("----- gold -----")
print(ex["transcription"], "|", ex["glosses"])

## 4. Smoke test: 10 dev rows, both settings

This only checks the pipeline. It is **not** a result.

In [ ]:
pd.set_option("display.max_colwidth", 80)
smoke = rz.main(["--lang", LANG, "--split", "dev", "--limit", "10"])
for p in smoke:
    df = pd.read_csv(p, keep_default_na=False)
    meta = json.load(open(p.with_suffix(".meta.json")))
    print("
=====", p.name, "=====")
    print(df.parse_status.value_counts().to_dict(), "| truncated:", int(df.truncated.sum()),
          "| seconds:", meta["runtime_seconds"], "| mean new tokens:", round(df.n_new_tokens.mean()))
    display(df[["gold_transliteration", "pred_transliteration", "gold_gloss", "pred_gloss", "parse_status"]])
    display(df[["raw_output"]])

## 5. Full run on the test split

Run this only after the smoke test looks right. Do not change prompts after looking at test outputs.

In [ ]:
full = rz.main(["--lang", LANG, "--split", "test"])
for p in full:
    df = pd.read_csv(p, keep_default_na=False)
    meta = json.load(open(p.with_suffix(".meta.json")))
    print(p.name, "| rows:", len(df), "|", df.parse_status.value_counts().to_dict(),
          "| truncated:", int(df.truncated.sum()), "| minutes:", round(meta["runtime_seconds"] / 60, 1))

## 6. Download the results

In [ ]:
out_dir = L["output_dir"]
!zip -r /content/zero_shot_{LANG}_results.zip {out_dir}
from google.colab import files
files.download(f"/content/zero_shot_{LANG}_results.zip")